# Phase 2: Hybrid Weapon Detection Training — Version 7 (Optimized)

This version incorporates optimizations based on early training logs from v6:
1. **Cosine Annealing Scheduler**: For smoother convergence and better global minimum search.
2. **Differential Learning Rate**: Dropping LR during Phase 2 (unfreeze) to preserve pretrained features.
3. **Enhanced Augmentations**: Increased probability for CCTV-like noise (Blur, ToGray).
4. **Sanitized Data Handling**: Assumes a cleaned dataset (duplicates/segments removed).

| Component | Location | Speed |
|-----------|----------|-------|
| Source Code & Model Weights | Google Drive (persistent) | — |
| Dataset (41k images, 10 GB) | `yolo_dataset.zip` on GDrive → extracted to Colab SSD | ⚡ ~100 MB/s |
| Training I/O | Colab local SSD (`/content/`) | ⚡ Native |
| Checkpoints | Google Drive `models/weights/` | Auto-saved |


## Step 1 — Environment & Dependencies
Install core packages and fix the **Numpy < 2.0** binary incompatibility.

In [1]:
# ── Install packages ────────────────────────────────────────────────────────
%pip install -q ultralytics albumentations timm

# ── Force Numpy < 2.0 ───────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2.0"], check=True)

import numpy as np, torch, os, sys, time, shutil, yaml
from pathlib import Path
from google.colab import drive

print(f"✅ Numpy  : {np.__version__}")
print(f"✅ PyTorch: {torch.__version__}")

if np.__version__.startswith('2.'):
    print("\n⚠️  [ACTION REQUIRED] Numpy 2.x still present.")
    print("    Click 'RESTART SESSION' in the Colab popup, then re-run from Step 2.")
else:
    print("\n✅ Environment ready. Proceed to Step 2.")

if torch.cuda.is_available():
    print(f"✅ GPU    : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU! Go to Runtime > Change runtime type > T4 GPU")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
✅ Numpy  : 1.26.4
✅ PyTorch: 2.10.0+cu128

✅ Environment ready. Proceed to Step 2.
✅ GPU    : Tesla T4
✅ VRAM   : 14.6 GB


## Step 2 — Configuration & Dataset Setup

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║              USER-EDITABLE CONFIGURATION — change these paths            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

ZIP_GDRIVE_PATH = "/content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System/yolo_dataset.zip"
GDRIVE_PROJECT_PATH = "/content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System/"
LOCAL_DATASET_DIR = "/content/yolo_dataset"

# ── 1. Mount Drive ────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── 2. Align project root ─────────────────────────────────────────────────
PROJECT_ROOT = Path(GDRIVE_PROJECT_PATH)
os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print(f"✅ Project root: {PROJECT_ROOT}")

# ── 3. Extract Dataset ────────────────────────────────────────────────────
DATA_YAML_PATH = Path(LOCAL_DATASET_DIR) / "yolo_dataset"/"data.yaml"
if not DATA_YAML_PATH.exists():
    if not os.path.exists(ZIP_GDRIVE_PATH):
        raise FileNotFoundError(f"❌ ZIP not found at: {ZIP_GDRIVE_PATH}")

    zip_size_gb = os.path.getsize(ZIP_GDRIVE_PATH) / (1024**3)
    print(f"📦 Found ZIP: {zip_size_gb:.1f} GB")

    LOCAL_ZIP = "/content/yolo_dataset.zip"
    print("📋 Copying ZIP to Colab SSD...")
    t0 = time.time()
    shutil.copy2(ZIP_GDRIVE_PATH, LOCAL_ZIP)
    dt = time.time() - t0
    print(f"✅ Copy done in {dt:.0f}s ({zip_size_gb/dt*1024 if dt > 0 else 0:.0f} MB/s)")

    print("📂 Extracting...")
    os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)
    !unzip -qo "{LOCAL_ZIP}" -d "{LOCAL_DATASET_DIR}"
    os.remove(LOCAL_ZIP)
    print("✅ Dataset extracted to SSD.")
else:
    print("✅ Dataset already present.")

from models.hybrid_model import HybridWeaponDetector
print("✅ HybridWeaponDetector ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Project root: /content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System
✅ Dataset already present.
✅ HybridWeaponDetector ready.


## Step 3 — Dataset Verification (Sanity Check)
Confirm images/labels count and `data.yaml` integrity.

In [3]:
import yaml
with open(DATA_YAML_PATH) as f: cfg = yaml.safe_load(f)

print("═" * 50)
print(f"Names : {cfg.get('names')}")
print("═" * 50)

base_path = Path(LOCAL_DATASET_DIR) / "yolo_dataset"

total = 0
for split in ['train', 'val']:
    img_dir = base_path / cfg[split]
    if img_dir.exists():
        n_img = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
        total += n_img
        print(f"✅ {split:5s}: {n_img:>6,} images")
    else:
        print(f"⚠️  {split:5s}: directory not found at {img_dir}")

print(f"\n📊 Total images: {total:,}")
if total > 40000: print("✅ Dataset looks complete!")
else: print("⚠️  Image count lower than expected. Check extraction logs.")

══════════════════════════════════════════════════
Names : {0: 'Weapon', 1: 'Person', 2: 'Confuser'}
══════════════════════════════════════════════════
✅ train: 40,669 images
✅ val  :  4,673 images

📊 Total images: 45,342
✅ Dataset looks complete!


## Step 4 — Optimized Hybrid Trainer (Phase-Aware Scheduler)

**V7 Updates**:
- **CosineAnnealingLR**: For smooth LR decay.
- **Differential Phase 2 LR**: LR drops to `1e-5` when unfreezing backbone.
- **Robust Resume**: Correctly reconstructs optimizer param groups when resuming in Phase 2.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
from tqdm import tqdm
from pathlib import Path
import shutil

def get_dataloaders(
    data_yaml_path: str,
    batch_size: int = 16,
    imgsz: int = 640,
    *,
    num_workers: int = 2,
    pin_memory: bool = True,
    persistent_workers: bool | None = None,
    prefetch_factor: int | None = 2,
    ):
    data_cfg = check_det_dataset(data_yaml_path)

    train_set = YOLODataset(
        img_path=data_cfg["train"], imgsz=imgsz,
        augment=True, batch_size=batch_size, task="detect", data=data_cfg,
    )
    val_set = YOLODataset(
        img_path=data_cfg["val"], imgsz=imgsz,
        augment=False, batch_size=batch_size, task="detect", data=data_cfg,
    )

    if persistent_workers is None:
        persistent_workers = num_workers > 0

    train_loader_kwargs = dict(
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=train_set.collate_fn,
    )
    val_loader_kwargs = dict(
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=val_set.collate_fn,
    )

    # Only valid when num_workers > 0
    if num_workers > 0:
        train_loader_kwargs["persistent_workers"] = persistent_workers
        val_loader_kwargs["persistent_workers"] = persistent_workers
        if prefetch_factor is not None:
            train_loader_kwargs["prefetch_factor"] = prefetch_factor
            val_loader_kwargs["prefetch_factor"] = prefetch_factor

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        **train_loader_kwargs,
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        **val_loader_kwargs,
    )
    return train_loader, val_loader

class HybridTrainer:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        device: str = "cuda",
        weights_dir: str = "models/weights",
        freeze_epochs: int = 10,
        *,
        val_every: int = 5,
        save_every: int = 5,
        sync_dir: str | None = None,
        sync_every: int = 5,
    ):
        self.device = device
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.scaler = GradScaler("cuda")
        self.weights_dir = Path(weights_dir)
        self.freeze_epochs = int(freeze_epochs)
        self.val_every = int(val_every)
        self.save_every = int(save_every)
        self.sync_dir = Path(sync_dir) if sync_dir else None
        self.sync_every = int(sync_every)
        self.best_loss = float("inf")
        self.criterion = model.head.compute_loss
        self.weights_dir.mkdir(parents=True, exist_ok=True)
        if self.sync_dir is not None:
            self.sync_dir.mkdir(parents=True, exist_ok=True)

    def _set_backbone_frozen(self, frozen: bool):
        for param in self.model.backbone.parameters():
            param.requires_grad = not frozen
        tag = "frozen ❄️" if frozen else "unfrozen 🔥"
        print(f"  Backbone {tag}")

    def _build_optimizer(self, lr=1e-4):
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
        return optimizer

    def _sync_files(self, filenames: list[str]):
        if self.sync_dir is None:
            return
        for name in filenames:
            src = self.weights_dir / name
            if src.exists():
                shutil.copy2(src, self.sync_dir / name)

    def run(self, total_epochs=50, resume=True):
        ckpt_path = self.weights_dir / "last.pt"
        ckpt = None
        ckpt_epoch = 0
        start_epoch = 1

        if resume and ckpt_path.exists():
            ckpt = torch.load(ckpt_path, map_location=self.device)
            ckpt_epoch = int(ckpt.get("epoch", 0))
            start_epoch = ckpt_epoch + 1
            print(f"Found checkpoint. Resuming from epoch {start_epoch}")

        start_in_phase2 = (start_epoch > self.freeze_epochs)
        ckpt_in_phase2 = (ckpt_epoch > self.freeze_epochs)

        # Build optimizer/scheduler for the phase we are STARTING in
        if start_in_phase2:
            self._set_backbone_frozen(False)
            optimizer = optim.AdamW(
                [
                    {"params": self.model.backbone.parameters(), "lr": 1e-5},
                    {"params": self.model.neck.parameters(), "lr": 5e-5},
                    {"params": self.model.head.parameters(), "lr": 5e-5},
                ],
                weight_decay=1e-4,
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=total_epochs - self.freeze_epochs,
            )
            is_phase2 = True
        else:
            self._set_backbone_frozen(True)
            optimizer = self._build_optimizer(lr=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=total_epochs,
            )
            is_phase2 = False

        # Restore checkpoint
        if ckpt is not None:
            self.model.load_state_dict(ckpt["model_state_dict"])
            self.best_loss = ckpt.get("best_loss", float("inf"))

            # Only restore optimizer/scheduler if the checkpoint phase matches the start phase
            if ckpt_in_phase2 == start_in_phase2:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                if "scheduler_state_dict" in ckpt:
                    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                    print("Scheduler state restored.")
                else:
                    print("No scheduler state in checkpoint; scheduler starts fresh.")
            else:
                print("Checkpoint saved in a different phase; skipping optimizer/scheduler restore.")

        for epoch in range(start_epoch, total_epochs + 1):
            # Phase transition (only when we are truly entering phase 2 during this run)
            if epoch == self.freeze_epochs + 1 and not is_phase2:
                print("\nPhase 2: Unfreezing (Lower LR)")
                self._set_backbone_frozen(False)
                optimizer = optim.AdamW(
                    [
                        {"params": self.model.backbone.parameters(), "lr": 1e-5},
                        {"params": self.model.neck.parameters(), "lr": 5e-5},
                        {"params": self.model.head.parameters(), "lr": 5e-5},
                    ],
                    weight_decay=1e-4,
                )
                scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=total_epochs - self.freeze_epochs,
                )
                is_phase2 = True

            self.model.train()
            avg_loss = 0.0
            lr_current = optimizer.param_groups[0]["lr"]
            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{total_epochs} [LR={lr_current:.2e}]")

            for batch in pbar:
                imgs = batch["img"].to(self.device, non_blocking=True).float() / 255.0
                optimizer.zero_grad(set_to_none=True)
                with autocast("cuda"):
                    preds = self.model(imgs)
                    loss = self.criterion(preds, batch, self.device)
                self.scaler.scale(loss).backward()
                self.scaler.step(optimizer)
                self.scaler.update()
                avg_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})

            avg_loss /= len(self.train_loader)

            # Validation (skip most epochs for speed)
            do_val = (epoch % self.val_every == 0) or (epoch == total_epochs) or (epoch == self.freeze_epochs)
            val_loss = float("nan")
            if do_val:
                self.model.eval()
                val_loss_acc = 0.0
                print("  Validating...")
                with torch.no_grad():
                    for batch in tqdm(self.val_loader, desc="Validation"):
                        imgs = batch["img"].to(self.device, non_blocking=True).float() / 255.0
                        with autocast("cuda"):
                            preds = self.model(imgs)
                            loss = self.criterion(preds, batch, self.device)
                        val_loss_acc += loss.item()
                val_loss = val_loss_acc / len(self.val_loader)

            scheduler.step()

            is_best = do_val and (val_loss < self.best_loss)
            if is_best:
                self.best_loss = val_loss

            # Checkpointing (save full state less often; sync to Drive less often)
            do_save = (epoch % self.save_every == 0) or (epoch == total_epochs) or (epoch == self.freeze_epochs) or (epoch == self.freeze_epochs + 1)
            if do_save:
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": self.model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "best_loss": self.best_loss,
                    },
                    self.weights_dir / "last.pt",
                )

            if is_best:
                torch.save(self.model.state_dict(), self.weights_dir / "best.pt")

            # Sync to Drive (optional)
            if self.sync_dir is not None:
                sync_last = do_save and ((epoch % self.sync_every == 0) or (epoch == total_epochs) or (epoch == self.freeze_epochs) or (epoch == self.freeze_epochs + 1))
                sync_best = is_best
                to_sync = []
                if sync_last:
                    to_sync.append("last.pt")
                if sync_best:
                    to_sync.append("best.pt")
                if to_sync:
                    self._sync_files(to_sync)

            if do_val:
                print(
                    f"Epoch {epoch:3d} | Train Loss: {avg_loss:.4f} | ",
                    f"Val Loss: {val_loss:.4f} | Best: {self.best_loss:.4f}",
                )
            else:
                print(f"Epoch {epoch:3d} | Train Loss: {avg_loss:.4f} | Val Loss: (skipped) | Best: {self.best_loss:.4f}")

## Step 5 — Launch Optimized Training

In [ ]:
# ── Speed knobs (edit these first) ──────────────────────────────────────────
BATCH_SIZE = 32      # try 24 → 32 on T4 (if no OOM)
IMG_SIZE   = 640     # keep 640 unless you also update anchors + Swin imgsz
NUM_WORKERS = 2     # try 2 / 4 / 6 depending on GPU utilization
VAL_EVERY  = 5       # validate every N epochs (cuts wall-clock a lot)
SAVE_EVERY = 5       # save full checkpoint every N epochs
SYNC_EVERY = 5       # sync checkpoints back to Drive every N epochs

from pathlib import Path
import shutil

# Save locally on Colab SSD (fast), sync to Drive periodically (persistent)
LOCAL_WEIGHTS_DIR = Path("/content/weights")
DRIVE_WEIGHTS_DIR = PROJECT_ROOT / "models" / "weights"
LOCAL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# If resuming, pull the latest checkpoint from Drive to SSD once
if (DRIVE_WEIGHTS_DIR / "last.pt").exists() and not (LOCAL_WEIGHTS_DIR / "last.pt").exists():
    shutil.copy2(DRIVE_WEIGHTS_DIR / "last.pt", LOCAL_WEIGHTS_DIR / "last.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

model = HybridWeaponDetector(backbone_variant="yolo11s.pt", nc=3, device=device)
model.head.alpha = torch.tensor([1.0, 1.0, 2.5], device=device)

train_loader, val_loader = get_dataloaders(
    str(DATA_YAML_PATH),
    batch_size=BATCH_SIZE,
    imgsz=IMG_SIZE,
    num_workers=NUM_WORKERS,
    prefetch_factor=2,
 )

trainer = HybridTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    weights_dir=str(LOCAL_WEIGHTS_DIR),
    freeze_epochs=10,
    val_every=VAL_EVERY,
    save_every=SAVE_EVERY,
    sync_dir=str(DRIVE_WEIGHTS_DIR),
    sync_every=SYNC_EVERY,
 )

trainer.run(total_epochs=50, resume=True)

Fast image access ✅ (ping: 0.0±0.0 ms, read: 161.9±104.2 MB/s, size: 174.4 KB)
Scanning /content/yolo_dataset/yolo_dataset/train/labels.cache... 41441 images, 3609 backgrounds, 3 corrupt: 100% ━━━━━━━━━━━━ 41444/41444 4.5Git/s 0.0s
/content/yolo_dataset/yolo_dataset/train/images/ds1_ds3_train_00943.jpg: corrupt JPEG restored and saved
/content/yolo_dataset/yolo_dataset/train/images/ds1_ds3_train_00944.jpg: corrupt JPEG restored and saved
/content/yolo_dataset/yolo_dataset/train/images/ds5_000000002444.jpg: ignoring corrupt image/label: cannot identify image file '/content/yolo_dataset/yolo_dataset/train/images/ds5_000000002444.jpg'
/content/yolo_dataset/yolo_dataset/train/images/ds5_000000002446.jpg: ignoring corrupt image/label: image file is truncated (13 bytes not processed)
/content/yolo_dataset/yolo_dataset/train/images/ds5_000000002459.jpg: ignoring corrupt image/label: cannot identify image file '/content/yolo_dataset/yolo_dataset/train/images/ds5_000000002459.jpg'
albumentation

Epoch 1/50 [LR=1.00e-04]:  16%|█▌        | 202/1296 [03:28<27:29,  1.51s/it, loss=4.6701]

## Step 6 — Visual Verification (Audit)
Run the trained model on a sample image to verify bounding box decoding logic.

In [ ]:
import cv2, glob
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

# 1. Locate a real val image using data.yaml (robust to extra folder levels)
with open(DATA_YAML_PATH) as f:
    cfg = yaml.safe_load(f)

dataset_root = Path(DATA_YAML_PATH).parent   # e.g. /content/yolo_dataset/yolo_dataset
val_img_dir = dataset_root / cfg["val"]      # e.g. val/images

candidates = sorted(
    glob.glob(str(val_img_dir / "*.jpg")) +
    glob.glob(str(val_img_dir / "*.png"))
)
if not candidates:
    raise FileNotFoundError(f"No validation images found in: {val_img_dir}")

sample_path = candidates[0]
sample_bgr = cv2.imread(sample_path)
if sample_bgr is None:
    raise FileNotFoundError(f"Failed to read image: {sample_path}")

# 2. Run prediction (predict() expects BGR ndarray)
model.eval()
detections = model.predict(sample_bgr, conf_threshold=0.1)

# 3. Convert to RGB for matplotlib display only
h, w = sample_bgr.shape[:2]
sample_rgb = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 10))
plt.imshow(sample_rgb)
ax = plt.gca()

for det in detections:
    x1 = det["bbox"][0] * w
    y1 = det["bbox"][1] * h
    x2 = det["bbox"][2] * w
    y2 = det["bbox"][3] * h
    rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="red", linewidth=2)
    ax.add_patch(rect)
    ax.text(
        x1, y1, f"{det['class_name']} {det['confidence']:.2f}",
        bbox=dict(facecolor="red", alpha=0.5), color="white"
    )

plt.axis("off")
plt.title(f"Visual Audit: Found {len(detections)} detections")
plt.show()